# Day 31 — Re-planning on failure

Plans meet reality and break. A robust agent notices a **failed step** and asks the
planner for a revised plan for the *remaining* work instead of charging ahead. Here we
force a failure and recover. > Needs a provider.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. The solution is below — try first.

In [ ]:
import json
from shared.llm import chat

def replan(goal, done, failure):
    raw = chat([
        {"role": "system", "content":
            "A step failed. Return ONLY a JSON array of revised remaining steps to still reach the goal."},
        {"role": "user", "content":
            f"Goal: {goal}\nCompleted: {done}\nFailure: {failure}"},
    ], temperature=0).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def execute(goal, steps):
    done = []
    for step in steps:
        # Simulate: any step containing "api" fails the first time.
        if "api" in step.lower() and step not in done:
            failure = f"network error on: {step}"
            # TODO: call replan(goal, done, failure) and RESTART execution with the new steps
            #       (return execute(goal, new_steps) once). Avoid infinite loops.
            raise NotImplementedError
        done.append(step)
    return done

# print(execute("Publish a post", ["Write draft", "Call publish api", "Confirm"]))

## 🔒 Solution

One correct way to do it.

In [ ]:
import json
from shared.llm import chat

def replan(goal, done, failure):
    raw = chat([
        {"role": "system", "content":
            "A step failed. Return ONLY a JSON array of revised remaining steps to still reach the goal."},
        {"role": "user", "content":
            f"Goal: {goal}\nCompleted: {done}\nFailure: {failure}"},
    ], temperature=0).strip()
    if raw.startswith("```"):
        raw = raw.strip("`").split("\n", 1)[-1]
    return json.loads(raw)

def execute(goal, steps, _replanned=False):
    done = []
    for step in steps:
        if "api" in step.lower() and not _replanned:
            new_steps = replan(goal, done, f"network error on: {step}")
            print("re-planned remaining steps:", new_steps)
            return done + execute(goal, new_steps, _replanned=True)
        done.append(step)
    return done

print(execute("Publish a post", ["Write draft", "Call publish api", "Confirm"]))